In [1]:
# --- FAST Melanoma Cancer Detection Model Training ---
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Enable mixed precision for faster training (if GPU available)
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("Mixed precision enabled for faster training")
except:
    print("Mixed precision not available, continuing with standard precision")

# Define paths
base_dir = './data'
original_train_dir = os.path.join(base_dir, 'original images/Train')
original_test_dir = os.path.join(base_dir, 'original images/Test')

# Check if directories exist
print("Checking directories...")
print("Original train exists:", os.path.exists(original_train_dir))
print("Original test exists:", os.path.exists(original_test_dir))

# Use original images
train_dir = original_train_dir
test_dir = original_test_dir

# Check class distribution
classes = sorted(os.listdir(train_dir))
print("Classes:", classes)
NUM_CLASSES = len(classes)
print(f"Number of classes: {NUM_CLASSES}")

# Count images in each class
class_counts = {}
for cls in classes:
    cls_path = os.path.join(train_dir, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        class_counts[cls] = count
        print(f"Class {cls}: {count} images")

# Image parameters (smaller for faster training)
IMG_SIZE = (128, 128)  # Reduced from 224x224 for faster training
BATCH_SIZE = 32
EPOCHS = 15  # Reduced from 30

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,  # Reduced from 20
    width_shift_range=0.1,  # Reduced from 0.2
    height_shift_range=0.1,  # Reduced from 0.2
    horizontal_flip=True,
    zoom_range=0.1,  # Reduced from 0.2
    validation_split=0.2
)

# Validation and test data generators (only rescaling)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    color_mode='rgb',
    classes=classes
)

val_generator = val_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    color_mode='rgb',
    classes=classes
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    color_mode='rgb',
    classes=classes
)

# Display class indices
class_indices = train_generator.class_indices
print("Class indices:", class_indices)

# Save class indices for later use
os.makedirs('models', exist_ok=True)
with open('models/class_indices.pkl', 'wb') as f:
    pickle.dump(class_indices, f)

# Build a SIMPLER and FASTER model
def create_fast_model(input_shape=(128, 128, 3), num_classes=9):
    # Use smaller EfficientNetB0 with pre-trained weights for faster convergence
    base_model = EfficientNetB0(
        weights='imagenet',  # Use pre-trained weights for faster training
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze the base model initially
    base_model.trainable = False
    
    # Add simpler custom layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)  # Reduced from 512
    x = Dropout(0.3)(x)  # Reduced from 0.5
    predictions = Dense(num_classes, activation='softmax')(x)
    
    # Create the model
    model = Model(inputs=base_model.input, outputs=predictions)
    
    return model, base_model

# Create model
print("Creating model...")
model, base_model = create_fast_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=NUM_CLASSES)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
print("Model architecture:")
model.summary()

# Define callbacks
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,  # Reduced from 5
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=2,  # Reduced from 3
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'models/melanoma_model_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Train the model - PHASE 1 (Frozen base model)
print("Starting Phase 1 training (frozen base model)...")
history = model.fit(
    train_generator,
    steps_per_epoch=min(50, train_generator.samples // BATCH_SIZE),  # Limit steps
    validation_data=val_generator,
    validation_steps=min(20, val_generator.samples // BATCH_SIZE),  # Limit steps
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

# Save the final model in .keras format
model.save('models/melanoma_model_final.keras')
print("Model saved as melanoma_model_final.keras")

# Plot training history
def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot accuracy
    ax1.plot(history.history['accuracy'])
    ax1.plot(history.history['val_accuracy'])
    ax1.set_title('Model Accuracy')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(['Train', 'Validation'], loc='upper left')
    
    # Plot loss
    ax2.plot(history.history['loss'])
    ax2.plot(history.history['val_loss'])
    ax2.set_title('Model Loss')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(['Train', 'Validation'], loc='upper left')
    
    plt.tight_layout()
    plt.savefig('models/training_history.png')
    plt.show()

plot_history(history)

# Evaluate the model on test data
print("Evaluating model on test data...")
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make predictions
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

# True classes
true_classes = test_generator.classes

# Classification report
print("Classification Report:")
print(classification_report(true_classes, predicted_classes, target_names=classes))

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('models/confusion_matrix.png')
plt.show()

# PHASE 2: Optional Fine-tuning (only if you want better performance)
fine_tune = input("Do you want to run fine-tuning? This will take more time. (y/n): ").lower().strip()

if fine_tune == 'y':
    print("Starting Phase 2 training (fine-tuning)...")
    
    # Unfreeze the base model
    base_model.trainable = True
    
    # Recompile with lower learning rate
    model.compile(
        optimizer=Adam(learning_rate=1e-5),  # Lower learning rate for fine-tuning
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Fine-tune for fewer epochs
    fine_tune_epochs = 5  # Reduced from 10
    
    history_fine = model.fit(
        train_generator,
        steps_per_epoch=min(50, train_generator.samples // BATCH_SIZE),
        validation_data=val_generator,
        validation_steps=min(20, val_generator.samples // BATCH_SIZE),
        epochs=len(history.history['loss']) + fine_tune_epochs,
        initial_epoch=len(history.history['loss']),
        callbacks=[early_stop, reduce_lr, checkpoint],
        verbose=1
    )
    
    # Save fine-tuned model
    model.save('models/melanoma_model_finetuned.keras')
    print("Fine-tuned model saved as melanoma_model_finetuned.keras")
    
    # Evaluate fine-tuned model
    test_loss_ft, test_accuracy_ft = model.evaluate(test_generator)
    print(f"After Fine-tuning - Test Accuracy: {test_accuracy_ft:.4f}")
    
    # Plot fine-tuning history
    plot_history(history_fine)
else:
    print("Skipping fine-tuning phase.")

# Save training history
history_df = pd.DataFrame(history.history)
history_df.to_csv('models/training_history.csv', index=False)

print("\n" + "="*50)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("="*50)
print("Models saved in .keras format:")
if os.path.exists('models/melanoma_model_best.keras'):
    print("- melanoma_model_best.keras (best validation accuracy)")
if os.path.exists('models/melanoma_model_final.keras'):
    print("- melanoma_model_final.keras (final epoch)")
if os.path.exists('models/melanoma_model_finetuned.keras'):
    print("- melanoma_model_finetuned.keras (fine-tuned)")
print("\nYou can now run the Streamlit app with: streamlit run app.py")

Mixed precision enabled for faster training
Checking directories...
Original train exists: True
Original test exists: True
Classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Number of classes: 9
Class actinic keratosis: 114 images
Class basal cell carcinoma: 376 images
Class dermatofibroma: 95 images
Class melanoma: 438 images
Class nevus: 357 images
Class pigmented benign keratosis: 462 images
Class seborrheic keratosis: 77 images
Class squamous cell carcinoma: 181 images
Class vascular lesion: 139 images
Found 1795 images belonging to 9 classes.
Found 444 images belonging to 9 classes.
Found 118 images belonging to 9 classes.
Class indices: {'actinic keratosis': 0, 'basal cell carcinoma': 1, 'dermatofibroma': 2, 'melanoma': 3, 'nevus': 4, 'pigmented benign keratosis': 5, 'seborrheic keratosis': 6, 'squamous cell carcinoma': 7, 'vascular lesion

ValueError: Shape mismatch in layer #1 (named stem_conv)for weight stem_conv/kernel. Weight expects shape (3, 3, 1, 32). Received saved weight with shape (3, 3, 3, 32)